<a href="https://colab.research.google.com/github/samradnyi-AIMLtech/RAG-Based-Document-Intelligence-System/blob/main/Copy_of_RAG_Document_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pypdf
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install transformers
!pip -q install accelerate
!pip -q install gradio

In [ ]:
import os
import re
import numpy as np
import faiss
import torch

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

pdf_filename = list(uploaded.keys())[0]

print("Uploaded file:", pdf_filename)

Saving trail pdf 1.pdf to trail pdf 1 (1).pdf
Uploaded file: trail pdf 1 (1).pdf


In [ ]:
def extract_text_from_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if text:
            pages.append({
                "page": page_number,
                "text": text
            })

    return pages


pages = extract_text_from_pdf(pdf_filename)

print("Number of pages:", len(pages))

Number of pages: 4


In [ ]:
for page in pages[:1]:

    print("\n")
    print("PAGE:", page["page"])
    print(page["text"][:2000])



PAGE: 1
Research on Current Technologies in 2026 
 
Introduction 
Technology is developing at an extraordinary speed and is changing the way people live, work, 
communicate, travel, and solve problems. In 2026, several technologies have moved beyond 
experimental research and are being used in businesses, healthcare, education, manufacturing, 
energy, and everyday life. Major areas of development include artificial intelligence, robotics, 
quantum computing, biotechnology, cybersecurity, advanced semiconductors, renewable 
energy, and space technology. The 2026 Stanford Emergi ng Technology Review identifies 
artificial intelligence, biotechnology, cybersecurity, energy, materials science, neuroscience, 
quantum technologies, robotics, semiconductors, and space as major frontier technology 
areas. (Stanford Emerging Technology Review) 
1. Artificial Intelligence 
Artificial Intelligence (AI) is currently one of the most influential technologies. Modern AI 
systems can understand lang

In [ ]:
def clean_text(text):

    text = re.sub(r'\s+', ' ', text)

    text = text.strip()

    return text

In [ ]:
for page in pages:

    page["text"] = clean_text(page["text"])

print("Text cleaning completed.")

Text cleaning completed.


In [ ]:
def create_chunks(pages, chunk_size=800, overlap=150):

    chunks = []

    for page in pages:

        text = page["text"]
        page_number = page["page"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunk_text = text[start:end]

            if chunk_text.strip():

                chunks.append({
                    "text": chunk_text,
                    "page": page_number
                })

            start += chunk_size - overlap

    return chunks

In [ ]:
chunks = create_chunks(
    pages,
    chunk_size=800,
    overlap=150
)

print("Total chunks:", len(chunks))

Total chunks: 18


In [ ]:
for i, chunk in enumerate(chunks[:5]):

    print("\n==============================")
    print("CHUNK:", i)
    print("PAGE:", chunk["page"])
    print(chunk["text"])


CHUNK: 0
PAGE: 1
Research on Current Technologies in 2026 Introduction Technology is developing at an extraordinary speed and is changing the way people live, work, communicate, travel, and solve problems. In 2026, several technologies have moved beyond experimental research and are being used in businesses, healthcare, education, manufacturing, energy, and everyday life. Major areas of development include artificial intelligence, robotics, quantum computing, biotechnology, cybersecurity, advanced semiconductors, renewable energy, and space technology. The 2026 Stanford Emergi ng Technology Review identifies artificial intelligence, biotechnology, cybersecurity, energy, materials science, neuroscience, quantum technologies, robotics, semiconductors, and space as major frontier technology areas. (Stanford E

CHUNK: 1
PAGE: 1
ty, energy, materials science, neuroscience, quantum technologies, robotics, semiconductors, and space as major frontier technology areas. (Stanford Emerging Techn

In [ ]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [ ]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

In [ ]:
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (18, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    embeddings.astype("float32")
)

print("FAISS vector database created!")
print("Number of vectors:", index.ntotal)

FAISS vector database created!
Number of vectors: 18


In [ ]:
def retrieve_documents(question, top_k=3):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        question_embedding.astype("float32"),
        top_k
    )

    results = []

    for distance, idx in zip(distances[0], indices[0]):

        if idx < len(chunks):

            results.append({
                "text": chunks[idx]["text"],
                "page": chunks[idx]["page"],
                "distance": float(distance)
            })

    return results

In [ ]:
question = "What is this document about?"

results = retrieve_documents(
    question,
    top_k=3
)

for result in results:

    print("\n==============================")
    print("PAGE:", result["page"])
    print("DISTANCE:", result["distance"])
    print(result["text"])


PAGE: 4
DISTANCE: 1.504544973373413
w emphasizes that major technologies increasingly intersect and influence one another. (Stanford Emerging Technology Review) These technologies offer enormous opportunities, but they also create risks involving privacy, employment, security, inequality, environmental impact, and ethics. Governments, researchers, businesses, and citizens must therefore ensure that technological p rogress is accompanied by responsible regulation, education, cybersecurity, and human oversight. If developed responsibly, current technologies can improve productivity, healthcare, scientific discovery, communication, and quality of life while helping society address some of its largest challenges.

PAGE: 3
DISTANCE: 1.5805747509002686
iable. 8. Space Technology

PAGE: 1
DISTANCE: 1.5875651836395264
Research on Current Technologies in 2026 Introduction Technology is developing at an extraordinary speed and is changing the way people live, work, communicate, travel, and solv

In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

print("LLM loaded!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded!


In [ ]:
def create_prompt(question, retrieved_documents):

    context = ""

    for i, doc in enumerate(retrieved_documents):

        context += f"""
Source {i + 1}
Page: {doc['page']}

{doc['text']}

"""

    prompt = f"""
You are a document intelligence assistant.

Answer the question using ONLY the information
provided in the context.

If the answer is not available in the context,
say:

"I could not find the answer in the document."

Do not invent information.

CONTEXT:

{context}

QUESTION:

{question}

ANSWER:
"""

    return prompt

In [ ]:
def generate_answer(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
def ask_document(question, top_k=3):

    retrieved_documents = retrieve_documents(
        question,
        top_k=top_k
    )

    prompt = create_prompt(
        question,
        retrieved_documents
    )

    answer = generate_answer(prompt)

    return answer, retrieved_documents

In [ ]:
question = input("Enter your question: ")

answer, sources = ask_document(
    question,
    top_k=3
)

print("\n==============================")
print("ANSWER")
print("==============================")

print(answer)

print("\n==============================")
print("SOURCES")
print("==============================")

for source in sources:

    print(
        f"Page {source['page']}"
    )

Enter your question: What is this document about

ANSWER
Science/Tech

SOURCES
Page 4
Page 1
Page 3


In [ ]:
def chat_with_document(question):

    answer, sources = ask_document(
        question,
        top_k=3
    )

    source_pages = sorted(
        set(
            source["page"]
            for source in sources
        )
    )

    return answer, source_pages

In [ ]:
answer, pages_used = chat_with_document(
    "What is the topic disscused in this document?"
)

print("ANSWER:")
print(answer)

print("\nSOURCE PAGES:")
print(pages_used)

ANSWER:
I could not find the answer in the document.

SOURCE PAGES:
[1, 4]


In [ ]:
import gradio as gr

In [ ]:
def chatbot(question):

    if not question.strip():

        return "Please enter a question.", ""

    answer, pages_used = chat_with_document(
        question
    )

    source_text = ", ".join(
        [f"Page {page}" for page in pages_used]
    )

    return answer, source_text

In [ ]:
demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="Ask a question about your document",
        placeholder="Example: What is this document about?"
    ),
    outputs=[
        gr.Textbox(
            label="Answer"
        ),
        gr.Textbox(
            label="Sources"
        )
    ],
    title="RAG Document Intelligence System",
    description="Ask questions about your uploaded PDF."
)

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://efc105f9cf3a3ed12e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
